In [172]:
import numpy as np
import scipy as sc

In [173]:
# Initialisation

N = 4 # Longueur du MPS
d = 2 # Nombre de paramètres physiques
chi = 5 # Bond dimension. On le fixe arbitrairement à 5 mais on pourrait l'augmenter ?

A = [] # liste contenant le MPS 

#Convention: on numérote initialement les tenseurs dans le sens anti trigo, en partant de la branche à gauche

#Ajout du premier tenseur (rang 2)
A.append(np.random.rand(d,chi))

#Ajout des tenseurs "centraux" (rang 3)
for _ in range(N-2):
    A.append(np.random.rand(chi,d,chi))

#Ajout du dernier tenseur (rang 2)
A.append(np.random.rand(chi,d))






In [174]:
np.shape(A[3])

(5, 2)

In [175]:
#Pour les tests, on prend H nul

#Ajout du premier MPO (rang 3)
H.append(np.zeros((d,1,d)))

#Ajout des MPO "centraux" (rang 3)
for _ in range(N-2):
    H.append(np.zeros((1,d,1,d)))

#Ajout du dernier MPO (rang 2)
H.append(np.zeros((1,d,d)))



Nous allons d'abord écrire les fonctions qui permettent de décomposer un hamiltonien sous forme d'un MPO

La fonction ci-dessous est pour l'instant incomplète

In [176]:
def contraction(H): #Si H est un MPO (une liste de tenseurs) cette fonction contracte le MPO. Sert pour les vérifications
    n=len(H)
    if n>=2:
        t=H[0]
        for i in range(n-1):
            t=np.tensordot(t,H[i+1],([i+1],[0])) #On doit décaler les indices de contraction car à chaque étape on "gagne" 2 ordres
            t=np.transpose(t,())
        return t
    else:
        return H

Pour construire H sous forme de MPO, nous utilisons la décomposition de Jordan-Wigner qui nous donne le hamiltonien décomposé en matrices de Pauli. Les matrices de Pauli sont les

In [177]:
Sx=np.array([[0,1],[1,0]], dtype="complex_")
Sy=np.array([[0,-1j],[1j,0]], dtype="complex_")
Sz=np.array([[1,0],[0,-1]], dtype="complex_")
I2=np.array([[1,0],[0,1]], dtype="complex_")

In [178]:
Splus=0.5*(Sx+1j*Sy)
Sminus=0.5*(Sx-1j*Sy)

In [179]:
Hx=np.zeros((1,2,1,2),dtype="complex_")
Hx[0,:,0,:]=Sx
Hxg=np.zeros((2,1,2),dtype="complex_")
Hxg[:,0,:]=Sx
Hxd=np.zeros((1,2,2),dtype="complex_")
Hxd[0,:,:]=Sx

In [180]:
Hy=np.zeros((1,2,1,2),dtype="complex_")
Hy[0,:,0,:]=Sy
Hyg=np.zeros((2,1,2),dtype="complex_")
Hyg[:,0,:]=Sy
Hyd=np.zeros((1,2,2),dtype="complex_")
Hyd[0,:,:]=Sy

In [181]:
Hz=np.zeros((1,2,1,2), dtype="complex_")
Hz[0,:,0,:]=Sz
Hzg=np.zeros((2,1,2),dtype="complex_")
Hzg[:,0,:]=Sz
Hzd=np.zeros((1,2,2),dtype="complex_")
Hzd[0,:,:]=Sz

In [182]:
H2=np.zeros((1,2,1,2),dtype="complex_")
H2[0,:,0,:]=I2
H2g=np.zeros((2,1,2),dtype="complex_")
H2g[:,0,:]=I2
H2d=np.zeros((1,2,2),dtype="complex_")
H2d[0,:,:]=I2

In [183]:
#Création du (MPO hamiltonien)

# prend deux MPO A et B en entrée (de même longueur) et renvoie le MPO correspondant à la somme de A et B
# Rq: les MPO sont supposés du même format que précedemment
def MPO_sum(A,B):
    n = len(A)
    chi_A = A[0].shape[1]
    chi_B = B[0].shape[1]
    chi_S = chi_A + chi_B
    S = []

    #Ajout du premier tenseur
    S_i = np.zeros((d,chi_S,d), dtype="complex_")
    for sigma1 in range(d):
        for sigma2 in range(d):

            for k in range(chi_A):
                S_i[sigma1,k,sigma2] = A[0][sigma1,k,sigma2]

            for k in range(chi_B):
                S_i[sigma1,k+chi_A,sigma2] = B[0][sigma1,k,sigma2]
            
    S.append(S_i)

    #Ajout des tenseurs "centraux"
    for i in range(1,n-1):
        S_i = np.zeros((chi_S,d,chi_S,d), dtype="complex_")
        for sigma1 in range(d):
            for sigma2 in range(d):
                
                for k in range(chi_A):
                    for l in range(chi_A):
                        S_i[k,sigma1,l,sigma2] = A[i][k,sigma1,l,sigma2]

                for k in range(chi_B):
                    for l in range(chi_B):
                        S_i[chi_A + k,sigma1,chi_A + l,sigma2] = B[i][k,sigma1,l,sigma2]      
        S.append(S_i)

    #Ajout du dernier tenseur
    S_i = np.zeros((chi_S,d,d), dtype="complex_")
    for sigma1 in range(d):
        for sigma2 in range(d):

            for k in range(chi_A):
                S_i[k,sigma1,sigma2] = A[n-1][k,sigma1,sigma2]

            for k in range(chi_B):
                S_i[k+chi_A,sigma1,sigma2] = B[n-1][k,sigma1,sigma2]
    S.append(S_i)
    
    
    return S


In [184]:
def pauli_to_MPO(H): #H est un hamiltonien donné sous forme d'une somme de produits de tenseurs de Pauli. Ici H est représenté sous forme de tableau où les termes en d'indice i dans la somme sont à la ième ligne
    M=H[0]
    for i in range(len(H)-1):
        M=MPO_sum(M,H[i+1])
    return M

In [185]:
H1=[[0.25*H2g,0.25*H2,0.25*H2,0.25*H2d],
            [0.25*Hzg,0.25*Hz,0.25*H2,0.25*H2d],
            [0.25*Hzg,0.25*H2,0.25*H2,0.25*H2d],
            [0.25*H2g,0.25*Hz,0.25*H2,0.25*H2d],
            [0.5*Hxg,0.5*Hz,0.5*Hx,0.5*H2d],
            [0.5*Hyg,0.5*Hz,0.5*Hy,0.5*H2d],
            [0.5*H2g,0.5*Hx,0.5*Hz,0.5*Hxd],
            [0.5*H2g,0.5*Hy,0.5*Hz,0.5*Hyd],
            [-0.5*H2g,-0.5*H2,-0.5*Hz,-0.5*H2d],
            [-0.5*H2g,-0.5*H2,-0.5*H2,-0.5*Hzd]]

In [186]:
Hid=[H2g,H2,H2,H2d]

In [187]:
H0=[np.zeros((2,1,2),dtype="complex_"),np.zeros((1,2,1,2),dtype="complex_"),np.zeros((1,2,1,2),dtype="complex_"),np.zeros((1,2,2),dtype="complex_")]

In [188]:
np.shape(Hid[3])

(1, 2, 2)

In [189]:
H=pauli_to_MPO(H1)

In [190]:
H=Hid

In [191]:
H=H0

In [192]:
H

[array([[[0.+0.j, 0.+0.j]],
 
        [[0.+0.j, 0.+0.j]]]),
 array([[[[0.+0.j, 0.+0.j]],
 
         [[0.+0.j, 0.+0.j]]]]),
 array([[[[0.+0.j, 0.+0.j]],
 
         [[0.+0.j, 0.+0.j]]]]),
 array([[[0.+0.j, 0.+0.j],
         [0.+0.j, 0.+0.j]]])]

In [193]:
np.shape(H[2])

(1, 2, 1, 2)

In [194]:
np.shape(H[3])

(1, 2, 2)

In [195]:
len(H)

4

In [196]:

def MPS_orth_left(A): #Normalise le MPS à gauche en effectuant des décompositions QR successives
    M = A.copy()

    q,r = sc.linalg.qr(M[0], mode = "economic")
    M[0] = q
    M[1] = np.tensordot(r,M[1], axes = ([1],[0]))

    
    #m=np.tensordot(M[0],M[1],axes=([1],[0]))
    #m=np.reshape(m,(d,d*chi))
    #q,r = sc.linalg.qr(m, mode = "economic")
    #M[0]=q
    #M[1]=np.reshape(r,(d,d,chi))
    
    for i in range(1,N-1): #On ne s'occupe pas du dernier tenseur
        p=np.shape(M[i])[0]
        M[i] = np.reshape(M[i],(d*p,chi))
        q,r = sc.linalg.qr(M[i], mode = "economic")
        M[i] = np.reshape(q,(np.shape(q)[0]//d,d,np.shape(q)[1]))
        M[i+1] = np.tensordot(r,M[i+1], axes = ([1],[0]))

        #m=np.tensordot(M[i],M[i+1], axes = ([2],[0]))
        #m=np.reshape(m,(d^(i+1),d*chi))
        #q,r = sc.linalg.qr(m, mode = "economic")
        #M[i]=np.reshape(q,(d^(i),d,d^(i+1)))
        #M[i+1]=np.reshape(r,(d,d,d*d))
        
        

    return M

def MPS_orth_right(A):

    M = A.copy()
    
    r,q = sc.linalg.rq(M[N-1],mode = "economic")
    M[N-1]=q
    M[N-2] = np.tensordot(M[N-2],r,axes = ([2],[0]))
    

    for i in range(N-2,0,-1): #On ne s'occupe pas du premier tenseur
        M[i]=np.reshape(M[i],(chi,d*np.shape(M[i])[2]))
        r,q = sc.linalg.rq(M[i], mode = "economic")
        M[i] = np.reshape(q,(np.shape(q)[0],d,np.shape(q)[1]//d))
        M[i-1] = np.tensordot(M[i-1],r,1)

    
    return M

print(MPS_orth_right(A))

[array([[-1.91615074e-02, -3.96000616e-01, -6.51603412e-02,
         7.92960649e-01, -1.86436026e+01],
       [-1.97884475e-02, -1.64569974e-01, -1.05087862e+00,
         1.95801394e+00, -2.04317088e+01]]), array([[[ 0.18339149,  0.34684031,  0.17648512, -0.22537502],
        [ 0.27241877,  0.76761982, -0.27436103,  0.15938104]],

       [[ 0.00532888,  0.00453076,  0.71434125, -0.39172257],
        [-0.10411727, -0.40641366, -0.28903606,  0.27688766]],

       [[ 0.00715348,  0.13619569, -0.63607679, -0.19173177],
        [-0.06565379, -0.23168844, -0.52255296,  0.45715781]],

       [[ 0.01714354,  0.08160531,  0.16594131,  0.63229057],
        [-0.02712601, -0.05587818, -0.67301598, -0.33001528]],

       [[-0.00456828, -0.04007797, -0.15794889, -0.59679689],
        [-0.0062464 , -0.04544586, -0.23341223, -0.74878094]]]), array([[[ 0.78515838, -0.14494271],
        [-0.58669081,  0.1353212 ]],

       [[ 0.55892637, -0.1856197 ],
        [ 0.80635656,  0.05418251]],

       [[-0.26

Dans les fonctions ci-dessous, la convention sur la numérotation des branches n'a pas toujours été respectée pour éviter de transposer à chaque étape

In [197]:

A = MPS_orth_right(A)


def right_contraction(Hr, Mt, Mb, H):
    
    Taux = np.tensordot(Hr,Mb, axes = ([2],[2]))
    Taux = np.tensordot(Taux,H, axes = ([3,0],[3,2]))
    Taux = np.tensordot(Taux,Mt, axes = ([0,3],[0,1]))
    Taux = np.transpose(Taux,(1,2,0))  #ordre final: de bas en haut: 2,0,1 (sens horaire en partant du milieu)

    return Taux

def left_contraction(Hl,Mt,Mb,H):

    Taux = np.tensordot(Hl,Mb, axes = ([2],[0]))
    Taux = np.tensordot(Taux,H, axes = ([1,2],[0,3]))
    Taux = np.tensordot(Taux,Mt, axes = ([0,2],[2,1]))
    Taux = np.transpose(Taux,(2,1,0)) #ordre final: de haut en bas: 0, 1, 2

    return Taux

#Calcul de R[i], composante à droite de la matrice effective (Rq: on ne calcule pas R[0] ici)

R = [0 for _ in range(N)] #liste contenant les R_i
L = [0 for _ in range(N)]


Taux = np.tensordot(H[N-1],A[N-1], axes = ([2],[1]))
Taux = np.tensordot(Taux,A[N-1].conj().T, axes = ([1],[0]))
R[N-2] = np.transpose(Taux,(0,2,1))
print(R[N-2].shape)

Taux = np.tensordot(H[0],A[0], axes = ([2],[0]))
Taux = np.tensordot(Taux,A[0].conj().T, axes = ([0],[1]))
L[1] = np.transpose(Taux,(2,0,1))
print(L[1].shape)


for i in range(N-3,-1,-1):
    R[i] = right_contraction(R[i+1],A[i+1].conj().T,A[i+1],H[i+1])

#for i in range(2,N):
#    L[i] = left_contraction(L[i-1],A[i-1].conj().T,A[i-1],H[i-1])

(1, 2, 2)
(5, 1, 5)


In [198]:
print(L[1].shape)
print(R[1].shape)

def effective_Matrix(L,R,H): #Calcule la matrice à diagonaliser
        Meff = np.tensordot(L,H,axes = ([1],[0]))
        Meff = np.tensordot(Meff,R, axes = ([3],[0]))
        Meff = np.transpose(Meff, (0,2,4,1,3,5))
        t=np.shape(Meff)
        Meff = np.reshape(Meff,(t[0]*t[1]*t[2],t[3]*t[4]*t[5]))
        return Meff,t

print(effective_Matrix(L[1],R[1],H[1]))
    

(5, 1, 5)
(1, 4, 4)
(array([[0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
       ...,
       [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]]), (5, 2, 4, 5, 2, 4))


In [200]:
def DMRG(Np):
    E = [] #liste des énergies

    for _ in range(Np):

        #Balayage de gauche à droite
       
        #Traitement du 1er tenseur (cas de bord: les dimensions de A[0] sont différentes)

        Meff = np.tensordot(H[0],R[0],axes = ([1],[0]))
        Meff = np.transpose(Meff, (0,2,1,3))
        t=np.shape(Meff)
        Meff = np.reshape(Meff, (t[0]*t[1],t[2]*t[3]))

        # Diagonalisation (Lanczos)
        val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA',, v0=A[0]) 
        vec = np.reshape(vec,(t[2],t[3]))
        E.append(val[0])

        #Décomposition SVD sur la matrice obtenue
        U,s,V = sc.linalg.svd(vec,full_matrices = False)
        S = np.diag(s)
        V = np.matmul(S,V)

        # Mise à jour des tenseurs A[0] et A[1]
        
        A[0] = U.copy()
        A[1] = np.tensordot(V,A[1],1)

        #Calcul de L[1] 

        Taux = np.tensordot(H[0],A[0], axes = ([2],[0]))
        Taux = np.tensordot(Taux,A[0].conj().T, axes = ([0],[1]))
        L[1] = np.transpose(Taux,(2,0,1))
        

        
        
        for i in range(1,N-1): #balayage de gauche à droite (pour les tenseurs centraux)
            Meff,t = effective_Matrix(L[i],R[i],H[i])

            # Diagonalisation (Lanczos)
            val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[i])
            vec = np.reshape(vec,(t[3]*t[4],t[5]))
            E.append(val[0])

            #Décomposition SVD sur la matrice obtenue
            U,s,V = sc.linalg.svd(vec,full_matrices = False)
            S = np.diag(s)
            V = np.matmul(S,V)
        
            # Mise à jour des tenseurs A[i] et A[i+1]
        
            A[i] = np.reshape(U,(t[3],t[4],t[5]))
            A[i+1] = np.tensordot(V,A[i+1],1)

            #Calcul de L[i] 
            L[i+1] = left_contraction(L[i],A[i].conj().T,A[i],H[i])

        #Traitement du dernier tenseur


        #Balayage de droite à gauche
        
        #Traitement du dernier tenseur
        
        Meff = np.tensordot(L[N-1],H[N-1],axes = ([1],[0]))
        Meff = np.transpose(Meff, (0,2,1,3))
        t=np.shape(Meff)
        Meff = np.reshape(Meff, (t[0]*t[1],t[2]*t[3]))

        # Diagonalisation (Lanczos)
        val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[0]) 
        vec = np.reshape(vec,(t[2],t[3]))
        E.append(val[0])

        # Décomposition SVD sur la matrice obtenue
        U,s,V = sc.linalg.svd(vec,full_matrices = False)
        S = np.diag(s)
        V = np.matmul(S,V)

        # Mise à jour des tenseurs A[N-1] et A[N-2]
        
        A[N-1] = V
        A[N-2] = np.tensordot(A[N-2],U,1)
    

        #Calcul de R[N-2] 
        Taux = np.tensordot(H[N-1],A[N-1], axes = ([2],[1]))
        Taux = np.tensordot(Taux,A[N-1].conj().T, axes = ([1],[0]))
        R[N-2] = np.transpose(Taux,(0,2,1))

        

        for i in range(N-2,0,-1): #balayage de droite à gauche (pour les MPS centraux)
    
            Meff,t = effective_Matrix(L[i],R[i],H[i])

            # Diagonalisation (Lanczos)
            val,vec = sc.sparse.linalg.eigsh(Meff, k=1, which='SA', v0=A[i]) 
            vec = np.reshape(vec,(t[3],t[4]*t[5]))
            E.append(val[0])

            #Décomposition SVD sur la matrice obtenue
            U,s,V = sc.linalg.svd(vec,full_matrices = False)
            S = np.diag(s)
            U = np.matmul(U,S)
        
            # Mise à jour des tenseurs A[i] et A[i+1]
        
            A[i] = np.reshape(V,(t[3],t[4],t[5]))
            A[i-1] = np.tensordot(A[i-1],U,1)

            #Calcul de R[i-1] 
            R[i-1] = right_contraction(R[i],A[i].conj().T,A[i],H[i])
        
    return E

print(DMRG(50))

        
        

ArpackError: ARPACK error -9: Starting vector is zero.